# Notebook 6 / 8 — End-to-End Workflow: Composing the Best Logics

> **Series.** This is the sixth of eight notebooks exploring how non-classical logics help agents communicate.
> 1. *Basics* — eight foundational logics
> 2. *Advanced* — fourteen rarer logics
> 3. *Synthesis* — cross-logic benchmarks and conclusions
> 4. *Applications* — ten general-purpose domains
> 5. *Language* — non-classical logics in NLP tasks
> 6. **Workflow** *(this notebook)* — a complete end-to-end pipeline that composes the best of the previous notebooks
> 7. *LangGraph* — the same pipeline rebuilt as a LangGraph state machine
> 8. *Experimental composition* — probing what happens when two non-classical logics overlap on the same linguistic phenomenon

## What this notebook is

Notebooks 1–5 looked at logics one at a time. This one **builds a single agent** that uses several of them together, in the order a real system would face them. The running scenario:

> A medical-information assistant receives a question from a user, retrieves passages from a knowledge store, fuses possibly conflicting evidence, checks norms ("is this advice we are allowed to give?"), and decides whether to answer, refuse, or escalate.

Each pipeline stage is a *thin* logic-specific module; the data flowing between them is the same record carrying *graded* and *uncertain* information all the way through. Nothing collapses to a single boolean until the final action.

## The composed logics — and what each stage owns

| Stage | Module | Logic | Why this logic? |
|---|---|---|---|
| 1. Intent parsing | `parse_intent` | **Default logic** | Conversational defaults ("medical question" implicates "wants safe advice") that may be overridden by explicit phrasing. |
| 2. Retrieval grounding | `retrieve_with_witness` | **Intuitionistic logic** | A claim is only assertible if a *retrieved passage* (constructive witness) supports it. |
| 3. Sense disambiguation | `disambiguate_terms` | **Dempster–Shafer** | Polysemous terms ("pressure", "depression") get belief mass over *sets* of senses, not a single label. |
| 4. Evidence fusion | `fuse_evidence` | **Possibilistic + LP** | Sources of mixed reliability are aggregated possibilistically; flat contradictions are tagged with LP value `B` instead of exploding. |
| 5. Annotator/source split | `aggregate_opinions` | **Subjective logic** | Disagreement among independent sources surfaces as residual `u`, routed to escalation. |
| 6. Norm check | `check_norms` | **SDL (deontic)** | Some answers are *forbidden* (e.g. dosing advice without a clinician). Distinguishes "should" from "is". |
| 7. Privacy / who-knows-what | `check_privacy` | **DEL (epistemic + action models)** | The assistant must not reveal what an upstream agent privately learned. Privacy is a checkable epistemic formula. |
| 8. Temporal protocol check | `check_protocol` | **LTL** | The full conversation trace must respect the protocol `request → (eventually answer ∨ escalate)`. |
| 9. Final decision | `decide` | **Bilattice (Belnap)** | All upstream signals collapse into one of `{T, F, N, B}`; the action depends on which value the controller sees. |

We deliberately omit a few logics covered earlier (CTL, ATL, IF, quantum, linear, free) — they belong to *other* workflows. The point of this notebook is to show that a small, well-chosen subset is enough to handle a realistic end-to-end task.

## How to read it

- Section A defines the shared `Claim` data structure that flows through every stage.
- Section B implements each module as a small function.
- Section C runs three full traces through the pipeline:
  - a clean question (assistant answers)
  - a contradictory-evidence question (assistant escalates)
  - a forbidden / private-data question (assistant refuses)
- Section D summarises which logic caught which failure mode.

## A. Shared data structure

Every stage reads and writes a `Claim`. It carries the *graded* state we need to keep until the very last decision.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Set, Tuple, Optional, FrozenSet
from itertools import product
from functools import reduce

Tv = frozenset({"T"}); Fv = frozenset({"F"}); Bv = frozenset({"T","F"}); Nv = frozenset()

@dataclass
class Opinion:
    """Subjective-logic opinion (b,d,u,a)."""
    b: float = 0.0
    d: float = 0.0
    u: float = 1.0
    a: float = 0.5
    def proj(self): return self.b + self.a*self.u

@dataclass
class Claim:
    text: str                                  # the claim under consideration
    senses: Dict[FrozenSet[str], float] = field(default_factory=dict)  # Dempster–Shafer mass
    witnesses: List[str] = field(default_factory=list)                 # intuitionistic evidence
    necessity: float = 0.0                     # possibilistic weight (0..1)
    lp_value: FrozenSet[str] = field(default_factory=lambda: frozenset())
    opinion: Opinion = field(default_factory=Opinion)
    obligations: Set[str] = field(default_factory=set)
    forbidden: bool = False
    privacy_violated: bool = False
    protocol_ok: bool = True

    def summary(self):
        return (f"  text='{self.text}'\n"
                f"  senses={ {tuple(sorted(s)):round(v,2) for s,v in self.senses.items()} }\n"
                f"  witnesses={self.witnesses}\n"
                f"  necessity={self.necessity:.2f}  lp={set(self.lp_value)}\n"
                f"  opinion=(b={self.opinion.b:.2f},d={self.opinion.d:.2f},u={self.opinion.u:.2f}) proj={self.opinion.proj():.2f}\n"
                f"  forbidden={self.forbidden}  privacy_violated={self.privacy_violated}  protocol_ok={self.protocol_ok}")

## B. The pipeline modules

Each function is one logic, kept small.

In [ ]:
# 1. Intent parsing — default logic.
# Defaults: medical question implicates 'wants_safe_advice'; an explicit 
# 'just curious' qualifier blocks it.
@dataclass
class Default:
    pre: str; justification: str; conclusion: str

DEFAULTS = [
    Default("medical_question", "safe_advice", "wants_safe_advice"),
]

def parse_intent(question: str) -> Set[str]:
    facts = set()
    if any(k in question.lower() for k in ["dose","symptom","medication","pressure","depression"]):
        facts.add("medical_question")
    if "just curious" in question.lower():
        facts.add("not_safe_advice")
    if "my friend" in question.lower() or "someone else" in question.lower():
        facts.add("third_party_question")
    # Apply defaults
    out = set(facts)
    changed = True
    while changed:
        changed = False
        for d in DEFAULTS:
            if d.pre in out and ("not_"+d.justification) not in out and d.conclusion not in out:
                out.add(d.conclusion); changed = True
    return out

In [ ]:
# 2. Retrieval grounding — intuitionistic.
# Toy 'knowledge base' of passages. A claim is only assertible if a passage backs it.
KB_PASSAGES = {
    "para_1": ("ibuprofen typical adult dose is 200-400 mg every 4-6 hours",
               {"ibuprofen_dose"}),
    "para_2": ("high blood pressure is defined as systolic >= 130 mmHg",
               {"hypertension_threshold"}),
    "para_3": ("depression screening uses the PHQ-9 questionnaire",
               {"depression_screening"}),
    "para_4": ("acetaminophen and ibuprofen can be alternated under clinician supervision",
               {"ibuprofen_combo", "clinician_required"}),
}

def retrieve_with_witness(target_atom: str) -> List[str]:
    """Return passage IDs that constructively support target_atom."""
    return [pid for pid,(_,atoms) in KB_PASSAGES.items() if target_atom in atoms]

In [ ]:
# 3. Sense disambiguation — Dempster-Shafer.
# 'pressure' may be {blood_pressure, atmospheric, social}; context cues narrow it.
def dempster(m1, m2):
    out, K = {}, 0.0
    for A,pA in m1.items():
        for B,pB in m2.items():
            inter = A & B
            if not inter: K += pA*pB
            else: out[inter] = out.get(inter,0) + pA*pB
    if K >= 1.0: return out, K
    return {s:v/(1-K) for s,v in out.items()}, K

fs = lambda *xs: frozenset(xs)
SENSE_PRIORS = {
    "pressure":   {fs("blood_pressure","atmospheric","social"): 1.0},
    "depression": {fs("clinical","economic","geographic"): 1.0},
}
CONTEXT_CUES = {
    "blood":    {fs("blood_pressure"): 0.8, fs("blood_pressure","social"): 0.2},
    "weather":  {fs("atmospheric"): 0.9, fs("atmospheric","social"): 0.1},
    "clinical": {fs("clinical"): 0.85, fs("clinical","economic"): 0.15},
    "market":   {fs("economic"): 0.9, fs("economic","geographic"): 0.1},
}

def disambiguate_terms(term: str, cues: List[str]) -> Dict[FrozenSet[str], float]:
    mass = SENSE_PRIORS.get(term, {fs("unknown"):1.0})
    for cue in cues:
        if cue in CONTEXT_CUES:
            mass, _ = dempster(mass, CONTEXT_CUES[cue])
    return mass

In [ ]:
# 4. Evidence fusion — possibilistic + LP.
# Each retrieved passage carries a source-reliability weight.
PASSAGE_RELIABILITY = {
    "para_1": 0.95, "para_2": 0.90, "para_3": 0.85, "para_4": 0.92,
}

def fuse_evidence(passage_ids: List[str], contradicts: bool=False) -> Tuple[float, FrozenSet[str]]:
    if not passage_ids:
        return 0.0, Nv
    weights = [PASSAGE_RELIABILITY[p] for p in passage_ids]
    # Possibilistic aggregation: best supporting source dominates.
    necessity = max(weights)
    # LP value: positive support, plus negative if contradicts==True.
    lp = Tv
    if contradicts: lp = frozenset(lp | Fv)   # becomes B = {T,F}
    return necessity, lp

In [ ]:
# 5. Source-opinion fusion — subjective logic.
def fuse_opinions(o1: Opinion, o2: Opinion) -> Opinion:
    if o1.u == 0 and o2.u == 0:
        return Opinion((o1.b+o2.b)/2, (o1.d+o2.d)/2, 0.0, o1.a)
    den = o1.u + o2.u - o1.u*o2.u
    return Opinion(
        (o1.b*o2.u + o2.b*o1.u)/den,
        (o1.d*o2.u + o2.d*o1.u)/den,
        (o1.u*o2.u)/den,
        o1.a,
    )

def aggregate_opinions(passage_ids: List[str], contradicts: bool) -> Opinion:
    if not passage_ids:
        return Opinion(0.0, 0.0, 1.0)
    ops = [Opinion(b=PASSAGE_RELIABILITY[p], d=0.0, u=1-PASSAGE_RELIABILITY[p]) for p in passage_ids]
    if contradicts:
        ops.append(Opinion(b=0.0, d=0.85, u=0.15))   # one dissenting source
    return reduce(fuse_opinions, ops)

In [ ]:
# 6. Norm check — SDL.
# Some answer types are forbidden (O ¬answer); others require a clinician disclaimer.
FORBIDDEN_INTENTS = {"unsupervised_dosing", "diagnose_third_party"}
OBLIGATION_TRIGGERS = {
    "medical_question": "add_disclaimer",
}

def check_norms(intent: Set[str], retrieved: List[str]) -> Tuple[bool, Set[str]]:
    forbidden = False
    obligations = set()
    if "third_party_question" in intent and "medical_question" in intent:
        forbidden = True
    if any("clinician_required" in KB_PASSAGES[p][1] for p in retrieved):
        forbidden = True
    for k in intent:
        if k in OBLIGATION_TRIGGERS:
            obligations.add(OBLIGATION_TRIGGERS[k])
    return forbidden, obligations

In [ ]:
# 7. Privacy check — DEL-flavoured epistemic gate.
# Some passages are tagged 'private to upstream agent'; revealing them would let the user
# infer private state. We model this as: if a private passage is in the retrieval set,
# the privacy formula ¬K_user private fails.
PRIVATE_PASSAGES = set()   # filled per-scenario

def check_privacy(retrieved: List[str]) -> bool:
    return any(p in PRIVATE_PASSAGES for p in retrieved)

In [ ]:
# 8. Protocol check — LTL.
# Trace records what happened at each step. The protocol must satisfy:
#   G(request -> F(answer ∨ escalate))
def ltl(f, trace, i=0):
    n = len(trace)
    if i >= n: return False
    op = f[0]
    if op=="atom": return f[1] in trace[i]
    if op=="not":  return not ltl(f[1],trace,i)
    if op=="and":  return ltl(f[1],trace,i) and ltl(f[2],trace,i)
    if op=="or":   return ltl(f[1],trace,i) or  ltl(f[2],trace,i)
    if op=="imp":  return (not ltl(f[1],trace,i)) or ltl(f[2],trace,i)
    if op=="F":    return any(ltl(f[1],trace,j) for j in range(i,n))
    if op=="G":    return all(ltl(f[1],trace,j) for j in range(i,n))
    raise ValueError(op)

PROTOCOL = ("G",("imp",("atom","request"),
                       ("F",("or",("atom","answer"),("atom","escalate")))))

def check_protocol(trace) -> bool:
    return ltl(PROTOCOL, trace)

In [ ]:
# 9. Final decision — bilattice collapse.
# Map all upstream signals into a Belnap value, then to an action.
def decide(claim: Claim) -> str:
    if claim.forbidden:           return "REFUSE (forbidden by norms)"
    if claim.privacy_violated:    return "REFUSE (would leak private info)"
    if not claim.protocol_ok:     return "PROTOCOL ERROR"
    if not claim.witnesses:       return "REFUSE (no constructive witness)"

    # Belnap summary of evidence
    if claim.lp_value == Bv:                         belnap = "B"
    elif claim.lp_value == Tv and claim.opinion.proj() > 0.7: belnap = "T"
    elif claim.lp_value == Fv:                       belnap = "F"
    else:                                            belnap = "N"

    if belnap == "T":
        suffix = " + DISCLAIMER" if "add_disclaimer" in claim.obligations else ""
        return f"ANSWER (cite {claim.witnesses}){suffix}"
    if belnap == "B":
        return "ESCALATE (sources disagree)"
    if belnap == "N":
        return "ESCALATE (insufficient evidence)"
    return "REFUSE (negative evidence dominates)"

## C. Three end-to-end runs

We now plug the modules into a single pipeline and run three contrasting questions through it. The same `Claim` object is enriched at every stage; nothing collapses to a single boolean until `decide()`.

In [ ]:
def pipeline(question: str, target_atom: str, cues: List[str],
             contradicts: bool=False, private_set: Set[str]=set()) -> Tuple[Claim, str]:
    global PRIVATE_PASSAGES
    PRIVATE_PASSAGES = private_set

    trace = [{"request"}]
    intent       = parse_intent(question)
    witnesses    = retrieve_with_witness(target_atom)
    senses       = disambiguate_terms("pressure" if "pressure" in question else
                                      "depression" if "depression" in question else "unknown",
                                      cues)
    necessity, lp = fuse_evidence(witnesses, contradicts=contradicts)
    opinion       = aggregate_opinions(witnesses, contradicts=contradicts)
    forbidden, obligations = check_norms(intent, witnesses)
    privacy_bad  = check_privacy(witnesses)

    claim = Claim(
        text=target_atom,
        senses=senses,
        witnesses=witnesses,
        necessity=necessity,
        lp_value=lp,
        opinion=opinion,
        obligations=obligations,
        forbidden=forbidden,
        privacy_violated=privacy_bad,
    )
    # Synthesise the final trace step from the (provisional) action
    action = decide(claim)
    if action.startswith("ANSWER"):  trace.append({"answer"})
    elif "ESCALATE" in action:        trace.append({"escalate"})
    else:                             trace.append({"refuse"})
    claim.protocol_ok = check_protocol(trace) or {"refuse"} in trace  # refuse also closes the protocol
    return claim, action

In [ ]:
print("\n══════════ RUN 1 — clean medical question ══════════")
q1 = "What is the typical adult dose of ibuprofen?"
claim, action = pipeline(q1, target_atom="ibuprofen_dose", cues=["clinical"])
print(f"Q: {q1}")
print(claim.summary())
print(f"\n → ACTION: {action}")

In [ ]:
print("\n══════════ RUN 2 — contradictory evidence ══════════")
q2 = "Are blood pressure readings above 130 already hypertension?"
claim, action = pipeline(q2, target_atom="hypertension_threshold",
                          cues=["blood","clinical"], contradicts=True)
print(f"Q: {q2}")
print(claim.summary())
print(f"\n → ACTION: {action}")

In [ ]:
print("\n══════════ RUN 3 — forbidden third-party dosing question ══════════")
q3 = "My friend wants to know how much acetaminophen to mix with ibuprofen."
claim, action = pipeline(q3, target_atom="ibuprofen_combo", cues=["clinical"])
print(f"Q: {q3}")
print(claim.summary())
print(f"\n → ACTION: {action}")

In [ ]:
print("\n══════════ RUN 4 — privacy violation ══════════")
# Suppose para_3 is privately tagged as 'belongs to an upstream patient record'.
q4 = "How is depression usually screened?"
claim, action = pipeline(q4, target_atom="depression_screening",
                          cues=["clinical"], private_set={"para_3"})
print(f"Q: {q4}")
print(claim.summary())
print(f"\n → ACTION: {action}")

## D. Which logic caught what?

| Run | Failure mode | Caught by | How |
|---|---|---|---|
| 1 | none — clean answer | intuitionistic + possibilistic + SDL | passage retrieved, high necessity, disclaimer obligation added |
| 2 | sources contradict | LP + bilattice + subjective | LP value `B`, opinion's `b ≈ d` → projection ≈ ½, decided to escalate |
| 3 | forbidden third-party dosing | default + SDL | default fired `wants_safe_advice`; norm check raised `forbidden` because `clinician_required` passage is in scope |
| 4 | private upstream data | DEL-style epistemic gate | a passage in `PRIVATE_PASSAGES` triggered `privacy_violated`, refusal precedes the bilattice collapse |

**Key composition principles demonstrated:**

1. **Carry the gradient.** No stage flattens its output. The `Claim` record has *six* logic-specific fields and only the last function (`decide`) collapses them into one of `{ANSWER, REFUSE, ESCALATE}`.
2. **Disjoint ownership.** Each logic owns one slice of state — defaults own intent, intuitionistic owns retrieval, DS owns senses, possibilistic+LP own evidence, subjective owns source disagreement, SDL owns norms, DEL owns privacy, LTL owns the trace, bilattice owns the final collapse. None of them touches another's slice.
3. **Refusal beats incorrect commitment.** Three of four runs refuse or escalate. A classical pipeline that forces a boolean at every step would have produced an answer (and likely a wrong one) for runs 2, 3, 4.
4. **Auditability is free.** Because the gradient is preserved, the system can explain *why* it refused — the controller has structured access to the necessity weight, the LP value, the opinion, the obligation list, and the privacy flag. A flat probability cannot give that.

## Variations to try

- Swap LP for bilattice end-to-end: gain the ability to distinguish *no information* (`N`) from *consistent positive evidence* (`T`).
- Replace SDL with **default + deontic** so the disclaimer obligation can be defeasibly cancelled (e.g. "professional context confirmed").
- Add an **AGM revision** stage between 4 and 5: when fresh contradictory evidence arrives, *revise* the prior belief instead of treating each query independently.
- Replace LTL with **CTL** when the conversation can branch (user picks between two clarifying follow-ups), and verify that *every* branch eventually closes the protocol.
- Wire **possibilistic** weights into the **subjective** opinions' uncertainty, so source reliability and dispersion of opinion are no longer two separate signals.

## Closing thought

After five notebooks of single-logic studies, the practical lesson is small but important: **you don't pick a logic, you pick a pipeline.** Each stage of a real agent has its own load-bearing detail — vague predicates here, contradictory sources there, norms somewhere else — and the right move is to choose the smallest non-classical fragment that captures *that* detail without leaking into the next stage.

The pipeline above uses nine fragments but no single one of them does anything fancy. The sophistication is in *not collapsing to booleans before you have to*.